In [0]:
dbutils.fs.ls("/Volumes/databricks_caresync/landing/landing_volume/neon_postgres/hospitals/2026-10-24_05:33:26/")

In [0]:
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
# landing_path = f'/Volumes/databricks_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/'

# landing_df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(landing_path) if source_file_format == 'csv' else spark.read.format('parquet').load(landing_path)

# display(landing_df)

if source_file_format == "csv":
    landing_df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(
            f"/Volumes/databricks_caresync/landing/landing_volume/"
            f"{source_system}/{table_name}/{landing_timestamp}/"
        )
    )
else:
    landing_df = spark.read.format("parquet").load(
        f"/Volumes/databricks_caresync/landing/landing_volume/"
        f"{source_system}/{table_name}/{landing_timestamp}/"
    )

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

display(updated_df)

In [0]:
updated_df.write.mode('append').saveAsTable(f'databricks_caresync.bronze.{table_name}')